# 17 — Deployable Operating Points, Label-Blind Acquisition, and Orientation Agreement

Three corrections to the experimental design, on the same corpora, caps and splits as notebooks 10 to 16, all at the reference seed.

Operating points. Earlier the retrained model was fitted and its threshold chosen on the same labelled buffer, which selects a decision rule on training rows. Here the buffer is split: most of it fits the model, the rest is reserved for choosing the threshold, and the total label count is unchanged, so the budget is honoured. Two analyses are then reported separately. The deployment-feasible one uses no evaluation labels and reports what the chosen threshold actually realises. The oracle diagnostic reports the highest true-positive rate attainable at a false-positive rate measured on the evaluation set itself, which allows strategies to be compared at a common alarm burden but is not a threshold rule an operator could follow.

Label-blind acquisition. The buffers used elsewhere are stratified by attack family with at least one row per family, which requires knowing families before paying for labels. That is an oracle-stratified reference, not an operational condition. A label-blind uniform sample is added as the deployment-feasible baseline, and iterative acquisition is re-initialised from it.

Orientation agreement. The paper defines inversion by which threshold orientation maximises MCC. The sign of the score-label covariance and the sign of AUROC minus one half are different quantities. All three are measured here for every transfer so their agreement can be reported rather than assumed.

A calibrated-then-rethresholded condition is also added, so that the contribution of probability estimation can be separated from that of decision-rule optimisation.

Results append to fc_results_v7.csv and fc_orientation.csv.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(corpora=['nf2018v2','nfunswv2','nftonv2','nfbotv2'], seed=42, test_size=0.30,
           train_cap=250_000, eval_cap=200_000, budgets=[0.0001, 0.001, 0.01],
           thr_frac=0.30, fpr_caps=[0.01, 0.001], ece_bins=15,
           rf_estimators=300, mlp_hidden=(128,64), mlp_max_iter=100)
MODELS = ['rf', 'lgbm', 'mlp']
V7_CSV  = f'{RESULT}/fc_results_v7.csv'
ORI_CSV = f'{RESULT}/fc_orientation.csv'
COLS = ['seed','source','target','model','budget','strategy','n_train','n_threshold','n_labels',
        'mcc','macro_f1','fp_rate','auprc_macro','thr',
        'tpr_dep','fpr_dep','tpr_oracle_fpr01','tpr_oracle_fpr001','fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label','Attack')]
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np, pandas as pd

def tpr_at_fpr(y, p, target_fpr):
    """Highest TPR attainable at or below target_fpr, with the threshold that attains it.
    Thresholds are evaluated at every distinct score, so the result is exact."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    order = np.argsort(-p, kind='mergesort'); ys, ps = y[order], p[order]
    P = int(ys.sum()); N = len(ys) - P
    if P == 0 or N == 0:
        return np.nan, np.nan
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last], fp[last], ps[last]
    ok = (fp / N) <= target_fpr
    if not ok.any():
        return 0.0, float(cuts[0]) + 1e-12
    i = int(np.argmax(np.where(ok, tp, -1)))
    return float(tp[i] / P), float(cuts[i])

def threshold_for_fpr_on_buffer(y_buf, p_buf, target_fpr):
    """Operating point an analyst could actually set: chosen on the labelled buffer only."""
    _, thr = tpr_at_fpr(y_buf, p_buf, target_fpr)
    return thr

def realised_at_threshold(y, p, thr):
    """TPR and FPR on the evaluation set at an externally chosen threshold."""
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= thr).astype(int)
    P = int(y.sum()); N = len(y) - P
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    return (tp / P if P else np.nan), (fp / N if N else np.nan)

def family_holdout_buffer(train_full, held_family, frac, seed, min_per_group=1):
    """Stratified buffer drawn only from families other than held_family, so the retrained
    model has never seen that attack type. Size matches the ordinary buffer at this budget."""
    pool = train_full[train_full['Attack'] != held_family]
    k = max(1, int(round(len(train_full) * frac)))
    parts = []
    for fam, g in pool.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * k / len(pool)))))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def eligible_families(df, min_rows=2000, max_share=0.60):
    """Attack families large enough to matter but not so dominant that holding one out
    leaves nothing to train on."""
    v = df.loc[df['Attack'] != 'Benign', 'Attack'].value_counts()
    n_att = int((df['Attack'] != 'Benign').sum())
    return [f for f, c in v.items() if c >= min_rows and c / n_att <= max_share]

def metrics_at_threshold(y_true, p_pos, thr):
    """Threshold-sensitive metrics taken at an externally chosen cut rather than at 0.5.
    Needed because the rethreshold strategy does not operate at 0.5, so reporting its
    macro-F1 or false-positive rate from the default cut would describe a different detector."""
    from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
    y = np.asarray(y_true).astype(int)
    pred = (np.asarray(p_pos) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(mcc=matthews_corrcoef(y, pred),
                macro_f1=f1_score(y, pred, average='macro'),
                fp_rate=fp / (fp + tn) if (fp + tn) else np.nan)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, matthews_corrcoef

def split_buffer(buf, thr_frac, seed, label_col='Label'):
    """Split a labelled buffer into a part used to fit the model and a part reserved for
    choosing the operating threshold. Both parts are drawn from the same budget, so the
    total label count is unchanged: thresholds are no longer selected on training rows."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(buf))
    thr_idx = []
    for _, g in buf.groupby(label_col, sort=True):
        pos = np.where(buf[label_col].values == g[label_col].iloc[0])[0]
        k = max(1, int(round(len(pos) * thr_frac))) if len(pos) > 1 else 0
        if k:
            thr_idx.extend(rng.choice(pos, size=min(k, len(pos) - 1), replace=False))
    thr_idx = np.array(sorted(set(thr_idx)), dtype=int)
    fit_idx = np.setdiff1d(idx, thr_idx)
    return buf.iloc[fit_idx], buf.iloc[thr_idx]

def orientation_signs(y, p):
    """The three quantities that must be distinguished: the sign of the score-label
    covariance, the sign of AUROC - 0.5, and which threshold orientation attains the higher
    MCC. They are not equivalent, so each is measured rather than inferred from another."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    # named cov_sy rather than cov: a column called 'cov' on a DataFrame is shadowed by
    # the DataFrame.cov method under attribute access, which silently yields the method
    out = dict(cov_sy=np.nan, auroc=np.nan, mcc_inc=np.nan, mcc_dec=np.nan)
    if len(np.unique(y)) < 2 or p.std() < 1e-12:
        return out
    out['cov_sy'] = float(np.cov(p, y, bias=True)[0, 1])
    out['auroc'] = float(roc_auc_score(y, p))
    order = np.argsort(-p, kind='mergesort'); ys = y[order]; ps = p[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp = tp[last], fp[last]
    def mcc(tp, fp, fn, tn):
        d = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        return np.where(d > 0, (tp * tn - fp * fn) / np.maximum(d, 1e-300), 0.0)
    out['mcc_inc'] = float(np.max(mcc(tp, fp, P - tp, N - fp)))
    tp2, fp2 = P - tp, N - fp
    out['mcc_dec'] = float(np.max(mcc(tp2, fp2, tp, fp)))
    return out

def tpr_at_common_fpr(y, p, cap):
    """Highest TPR attainable at or below a false-positive rate measured on the evaluation
    set itself. This is an oracle diagnostic, not a deployable threshold rule."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    P, N = int(y.sum()), int((1 - y).sum())
    if P == 0 or N == 0:
        return np.nan
    order = np.argsort(-p, kind='mergesort'); ys = y[order]; ps = p[order]
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp = tp[last], fp[last]
    ok = (fp / N) <= cap
    return float(tp[ok].max() / P) if ok.any() else 0.0

def uniform_buffer(train_full, frac, seed):
    """A label-blind sample: rows are drawn at random from the unlabelled pool with no use
    of family or class information, which is what an operator can actually do before paying
    for labels. The family-stratified buffer used elsewhere requires knowing attack families
    in advance and is therefore an oracle reference."""
    k = max(1, int(round(len(train_full) * frac)))
    return train_full.sample(n=min(k, len(train_full)), random_state=seed)

In [ ]:
from sklearn.model_selection import train_test_split

def record(row, path, cols):
    r = {c: row.get(c, np.nan) for c in cols}
    pd.DataFrame([r], columns=cols).to_csv(path, mode='a', index=False, header=not os.path.exists(path))

def key(src, tgt, m, b, s):
    return (src, tgt, m, f'{float(b):.6g}', s)

done = set()
if os.path.exists(V7_CSV):
    prev = pd.read_csv(V7_CSV)
    done = set(key(r.source, r.target, r.model, r.budget, r.strategy) for r in prev.itertuples())
    print(f'resume: {len(done)} rows recorded')

def is_done(*k):
    return key(*k) in done

def mark(src, tgt, m, b, s, metrics, **kw):
    record(dict(seed=CFG['seed'], source=src, target=tgt, model=m, budget=b, strategy=s,
                **metrics, **kw), V7_CSV, COLS)
    done.add(key(src, tgt, m, b, s))
    print(f"  {src}->{tgt} {m} b={b} {s}: MCC={metrics.get('mcc', float('nan')):.3f} "
          f"TPR_dep={kw.get('tpr_dep', float('nan')):.3f} FPR_dep={kw.get('fpr_dep', float('nan')):.3f}")

T0 = time.time()
def el(): return f'[{(time.time()-T0)/60:5.1f}m]'

seed = CFG['seed']
parts = {}
for tag, d in DATASETS.items():
    tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
    tr = tr.reset_index(drop=True)
    parts[tag] = dict(train=stratified_cap(tr, CFG['train_cap'], seed),
                      eval=stratified_cap(te, CFG['eval_cap'], seed).reset_index(drop=True),
                      train_full=tr)

# buffers: the oracle-stratified reference used elsewhere, and a label-blind uniform draw
BUF = {}
for tgt in CFG['corpora']:
    for b in CFG['budgets']:
        BUF[(tgt, b, 'stratified')] = stratified_frac(parts[tgt]['train_full'], b, seed)
        BUF[(tgt, b, 'labelblind')] = uniform_buffer(parts[tgt]['train_full'], b, seed)
        s_, u_ = BUF[(tgt, b, 'stratified')], BUF[(tgt, b, 'labelblind')]
        print(f"{tgt} b={b}: stratified {len(s_)} rows / {s_.Attack.nunique()} families, "
              f"label-blind {len(u_)} rows / {u_.Attack.nunique()} families, "
              f"attack rate {s_.Label.mean():.3f} vs {u_.Label.mean():.3f}")

# ---------- retraining with the threshold chosen on held-out buffer labels ----------
for tgt in CFG['corpora']:
    ev = parts[tgt]['eval']; yev = ev['Label'].values
    for mname in MODELS:
        for b in CFG['budgets']:
            for draw in ['stratified', 'labelblind']:
                s = f'retrain_{draw}'
                if is_done('any', tgt, mname, b, s):
                    continue
                buf = BUF[(tgt, b, draw)]
                if buf['Label'].nunique() < 2:
                    mark('any', tgt, mname, b, s, {'mcc': 0.0}, n_labels=len(buf)); continue
                fitb, thrb = split_buffer(buf, CFG['thr_frac'], seed)
                if fitb['Label'].nunique() < 2 or len(thrb) == 0:
                    mark('any', tgt, mname, b, s, {'mcc': 0.0}, n_labels=len(buf)); continue
                Xf, med = clean_X(fitb, FEATURES); Xt, _ = clean_X(thrb, FEATURES, medians=med)
                Xe, _ = clean_X(ev, FEATURES, medians=med)
                mdl = make_model(mname, seed, len(Xf))
                t0 = time.time(); mdl.fit(Xf, fitb['Label'].values); fs = round(time.time()-t0, 1)
                p_ev = mdl.predict_proba(Xe)[:, 1]
                p_thr = mdl.predict_proba(Xt)[:, 1]
                m = all_metrics(yev, p_ev)
                thr = threshold_for_fpr_on_buffer(thrb['Label'].values, p_thr, CFG['fpr_caps'][0])
                td, fd = (realised_at_threshold(yev, p_ev, thr) if not np.isnan(thr) else (np.nan, np.nan))
                mark('any', tgt, mname, b, s, m, n_train=len(fitb), n_threshold=len(thrb),
                     n_labels=len(buf), thr=thr, tpr_dep=td, fpr_dep=fd,
                     tpr_oracle_fpr01=tpr_at_common_fpr(yev, p_ev, 0.01),
                     tpr_oracle_fpr001=tpr_at_common_fpr(yev, p_ev, 0.001), fit_s=fs)
                del mdl, Xf, Xt, Xe; gc.collect()

# ---------- post-hoc strategies, with the same held-out thresholding, plus calibrate+rethreshold ----------
orows = []
for src in CFG['corpora']:
    Xs, med_s = clean_X(parts[src]['train'], FEATURES); ys = parts[src]['train']['Label'].values
    for mname in MODELS:
        src_model = make_model(mname, seed, len(Xs))
        t0 = time.time(); src_model.fit(Xs, ys)
        print(f'{el()} source fit {mname} on {src}: {time.time()-t0:.0f}s')
        for tgt in [x for x in CFG['corpora'] if x != src]:
            ev = parts[tgt]['eval']; yev = ev['Label'].values
            Xe, _ = clean_X(ev, FEATURES, medians=med_s)
            p_ev = src_model.predict_proba(Xe)[:, 1]

            o = orientation_signs(yev, p_ev)
            orows.append(dict(source=src, target=tgt, model=mname, **o,
                              tpr_oracle_fpr01=tpr_at_common_fpr(yev, p_ev, 0.01)))

            for b in CFG['budgets']:
                buf = BUF[(tgt, b, 'stratified')]
                if buf['Label'].nunique() < 2:
                    continue
                fitb, thrb = split_buffer(buf, CFG['thr_frac'], seed)
                Xf, _ = clean_X(fitb, FEATURES, medians=med_s)
                Xt, _ = clean_X(thrb, FEATURES, medians=med_s)
                p_fit = src_model.predict_proba(Xf)[:, 1]
                p_thr = src_model.predict_proba(Xt)[:, 1]
                yfit, ythr = fitb['Label'].values, thrb['Label'].values

                # the calibrated scores are needed by both conditions below, so the Platt fit is
                # not placed inside either resume guard: nesting them would leave the second
                # condition permanently unrun on any resume where only the first had completed
                need_cal = not is_done(src, tgt, mname, b, 'calibrate_split')
                need_cr  = not is_done(src, tgt, mname, b, 'calibrate_rethreshold')
                if (need_cal or need_cr) and len(np.unique(yfit)) > 1:
                    lr = platt_fit(p_fit, yfit)
                    pc_ev = platt_apply(lr, p_ev, yfit); pc_thr = platt_apply(lr, p_thr, yfit)

                    if need_cal:
                        m = all_metrics(yev, pc_ev)
                        thr = threshold_for_fpr_on_buffer(ythr, pc_thr, CFG['fpr_caps'][0])
                        td, fd = (realised_at_threshold(yev, pc_ev, thr) if not np.isnan(thr) else (np.nan, np.nan))
                        mark(src, tgt, mname, b, 'calibrate_split', m, n_train=len(fitb), n_threshold=len(thrb),
                             n_labels=len(buf), thr=thr, tpr_dep=td, fpr_dep=fd,
                             tpr_oracle_fpr01=tpr_at_common_fpr(yev, pc_ev, 0.01),
                             tpr_oracle_fpr001=tpr_at_common_fpr(yev, pc_ev, 0.001))

                    if need_cr and len(np.unique(ythr)) > 1:
                        bm, bt, bo = best_threshold_two_sided(ythr, pc_thr)
                        p_or = pc_ev if bo == 1 else 1.0 - pc_ev
                        m2 = all_metrics(yev, p_or); m2.update(metrics_at_threshold(yev, p_or, bt))
                        thr2 = threshold_for_fpr_on_buffer(ythr, pc_thr if bo == 1 else 1.0 - pc_thr,
                                                           CFG['fpr_caps'][0])
                        td2, fd2 = (realised_at_threshold(yev, p_or, thr2) if not np.isnan(thr2) else (np.nan, np.nan))
                        mark(src, tgt, mname, b, 'calibrate_rethreshold', m2, n_train=len(fitb),
                             n_threshold=len(thrb), n_labels=len(buf), thr=bt, tpr_dep=td2, fpr_dep=fd2,
                             tpr_oracle_fpr01=tpr_at_common_fpr(yev, p_or, 0.01),
                             tpr_oracle_fpr001=tpr_at_common_fpr(yev, p_or, 0.001))

                if not is_done(src, tgt, mname, b, 'rethreshold_split') and len(np.unique(ythr)) > 1:
                    bm, bt, bo = best_threshold_two_sided(ythr, p_thr)
                    p_or = p_ev if bo == 1 else 1.0 - p_ev
                    m = all_metrics(yev, p_or); m.update(metrics_at_threshold(yev, p_or, bt))
                    thr = threshold_for_fpr_on_buffer(ythr, p_thr if bo == 1 else 1.0 - p_thr, CFG['fpr_caps'][0])
                    td, fd = (realised_at_threshold(yev, p_or, thr) if not np.isnan(thr) else (np.nan, np.nan))
                    mark(src, tgt, mname, b, 'rethreshold_split', m, n_train=len(fitb), n_threshold=len(thrb),
                         n_labels=len(buf), thr=bt, tpr_dep=td, fpr_dep=fd,
                         tpr_oracle_fpr01=tpr_at_common_fpr(yev, p_or, 0.01),
                         tpr_oracle_fpr001=tpr_at_common_fpr(yev, p_or, 0.001))
                del Xf, Xt; gc.collect()
            del Xe, p_ev; gc.collect()
        del src_model; gc.collect()
    del Xs; gc.collect()

# merged rather than overwritten, so an interrupted run cannot truncate the file
OR = pd.DataFrame(orows)
if os.path.exists(ORI_CSV):
    OR = pd.concat([pd.read_csv(ORI_CSV), OR], ignore_index=True)
OR = OR.drop_duplicates(['source', 'target', 'model'], keep='last')
OR.round(6).to_csv(ORI_CSV, index=False)
print('saved', ORI_CSV, f'({len(OR)} orientation rows) | v7 rows recorded:', len(done))

In [ ]:
from scipy.stats import wilcoxon
CHR10 = chr(10)

V = pd.read_csv(V7_CSV).drop_duplicates(['source','target','model','budget','strategy'])
O = pd.read_csv(ORI_CSV)

print('=== 1. ORIENTATION: do the three definitions agree? ===')
O['s_cov'] = np.sign(O['cov_sy']); O['s_auc'] = np.sign(O.auroc - 0.5); O['s_mcc'] = np.sign(O.mcc_inc - O.mcc_dec)
n = len(O)
print(f"  cells: {n}")
print(f"  sign(Cov) == sign(AUROC-0.5):      {int((O.s_cov==O.s_auc).sum())}/{n} ({(O.s_cov==O.s_auc).mean():.0%})")
print(f"  sign(Cov) == best-MCC orientation: {int((O.s_cov==O.s_mcc).sum())}/{n} ({(O.s_cov==O.s_mcc).mean():.0%})")
print(f"  sign(AUROC-0.5) == best-MCC orient:{int((O.s_auc==O.s_mcc).sum())}/{n} ({(O.s_auc==O.s_mcc).mean():.0%})")
print(f"  all three agree:                   {int(((O.s_cov==O.s_auc)&(O.s_auc==O.s_mcc)).sum())}/{n}")
print('\n  inversion rate under each definition:')
print(f"    covariance negative        {(O.s_cov<0).mean():.0%}")
print(f"    AUROC below 0.5            {(O.s_auc<0).mean():.0%}")
print(f"    decreasing orientation wins {(O.s_mcc<0).mean():.0%}")
print('\n  by corpus pair (fraction of model cells inverted under each definition):')
print(O.groupby(['source','target'])[['s_cov','s_auc','s_mcc']].apply(lambda g:(g<0).mean()).round(2).to_string())

print('\n=== 2. DEPLOYMENT-FEASIBLE operating points (threshold from held-out buffer labels) ===')
dep = V[V.strategy.isin(['retrain_stratified','retrain_labelblind','calibrate_split',
                         'calibrate_rethreshold','rethreshold_split'])]
t = dep.groupby(['strategy','budget']).agg(mcc=('mcc','mean'), tpr=('tpr_dep','mean'),
        fpr=('fpr_dep','mean'), fpr_med=('fpr_dep','median'), n=('mcc','size')).round(4)
print(t.to_string())
print('\n  how often the realised FPR exceeds twice the 1% target:')
print(dep.groupby(['strategy','budget']).apply(lambda g:(g.fpr_dep>0.02).mean()).round(2).to_string())

print('\n=== 3. ORACLE DIAGNOSTIC: TPR at a common evaluation FPR (not a deployable rule) ===')
print(dep.groupby(['strategy','budget'])[['tpr_oracle_fpr01','tpr_oracle_fpr001']].mean().round(4).to_string())

print('\n=== 4. LABEL-BLIND vs ORACLE-STRATIFIED buffers ===')
a = V[V.strategy=='retrain_stratified'][['target','model','budget','mcc','tpr_dep','fpr_dep']].rename(
        columns={'mcc':'mcc_strat','tpr_dep':'tpr_strat','fpr_dep':'fpr_strat'})
bl = V[V.strategy=='retrain_labelblind'][['target','model','budget','mcc','tpr_dep','fpr_dep']].rename(
        columns={'mcc':'mcc_blind','tpr_dep':'tpr_blind','fpr_dep':'fpr_blind'})
C = a.merge(bl, on=['target','model','budget'])
# a label-blind buffer can contain a single class, in which case no model is fitted and
# tpr_dep is undefined. Averaging tpr_blind over the remaining cells would quietly drop
# exactly the cases where the label-blind draw fails worst, so those are counted first.
C['blind_degenerate'] = C.tpr_blind.isna()
print(C.groupby('budget')[['mcc_strat','mcc_blind']].mean().round(4).to_string())
print(CHR10 + '  single-class label-blind buffers (no model fitted), by budget and target:')
print(C[C.blind_degenerate].groupby(['budget','target']).size().to_string() if C.blind_degenerate.any()
      else '    none')
both = C[~C.blind_degenerate]
print(CHR10 + f"  operating points compared only where both conditions produced a model "
      f"({len(both)} of {len(C)} cells):")
print(both.groupby('budget')[['tpr_strat','tpr_blind','fpr_strat','fpr_blind']].mean().round(4).to_string())
for b, g in C.groupby('budget'):
    d = g.mcc_strat - g.mcc_blind
    p = wilcoxon(g.mcc_strat, g.mcc_blind).pvalue if len(g) > 5 else np.nan
    print(f"  budget {b}: oracle stratification is worth {d.mean():+.3f} MCC "
          f"(wins {int((d>0).sum())}/{len(d)}, p={p:.4f})")
print('\n  per target at the smallest budget:')
print(C[C.budget==CFG['budgets'][0]].groupby('target')[['mcc_strat','mcc_blind']].mean().round(3).to_string())

print('\n=== 5. does calibration plus rethresholding recover what rethresholding alone gives? ===')
cr = V[V.strategy=='calibrate_rethreshold'][['source','target','model','budget','mcc']].rename(columns={'mcc':'cal_rethr'})
rt = V[V.strategy=='rethreshold_split'][['source','target','model','budget','mcc']].rename(columns={'mcc':'rethr'})
ca = V[V.strategy=='calibrate_split'][['source','target','model','budget','mcc']].rename(columns={'mcc':'cal'})
K = cr.merge(rt, on=['source','target','model','budget']).merge(ca, on=['source','target','model','budget'])
print(K.groupby('budget')[['cal','cal_rethr','rethr']].mean().round(4).to_string())
d = (K.cal_rethr - K.rethr).abs()
print(f"  mean |calibrate+rethreshold - rethreshold| = {d.mean():.4f}, max {d.max():.4f}")
print("  (a monotone calibrator should leave the attainable decisions unchanged up to ties and orientation)")
V.round(5).to_csv(f'{RESULT}/fc_deployable_operating.csv', index=False)
print('\nsaved fc_deployable_operating.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "17: deployable operating points with held-out thresholding, label-blind acquisition, three-way orientation agreement"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)